# 02 - Materials, tallies and run settings

solution

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

---

# Exercises

## Exercise 1 - swap the coolant

Replace the coolant of the model (material 7) with a new material:

1. build material **99** from **natural** hydrogen and oxygen (zaids `1000` and `8000`) with
   atom fractions 2 and 1, on library `31c`, and give it a name;
2. add it to the material section;
3. make every cell that used material 7 use material 99 instead, at a mass density of
   **1.0 g/cc**.

**Hints**
* `Material.from_zaids(zaids, libman, lib, name=..., mat_id=...)`, where `zaids` is a list of
  `(zaid, fraction)` pairs;
* `inp.mat_section.append(...)`;
* `inp.replace_material(new_id, new_density, old_id)` - in MCNP a **mass** density is negative.

In [ ]:
from f4enix import Input
from f4enix.input.libmanager import LibManager
from f4enix.input.materials import Material

inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")
libman = LibManager()

# 1: natural zaids, the translation expands them with their natural abundances
water = Material.from_zaids(
    [('H', 2), ('O', 1)],
    libman,
    "31c",
    name="Light water, replaces M7",
    mat_id=99,
)

# 2: the new material becomes part of the model
inp.mat_section.append(water)

# 3: every cell that used material 7 now uses 99 at -1.0 g/cc
inp.replace_material(99, -1.0, 7)

print(water.to_text())

In [ ]:
# --- self-check, do not modify ---
names = [m.name for m in inp.mat_section]
assert "M99" in names, f"M99 was not added, materials are {names}"

m99 = inp.mat_section["M99"]
assert len(m99.zaids) == 5, (
    f"M99 has only {len(m99.zaids)} zaids - did you use the natural zaids 1000 and 8000?"
)

counts = inp.get_cells_summary()["material"].value_counts()
assert 7 not in counts.index, "there are still cells using material 7"
assert counts.get(99, 0) == 14, f"expected 14 cells with material 99, found {counts.get(99, 0)}"
assert float(inp.cells["34"].density) == -1.0, f"cell 34 density is {inp.cells['34'].density}"
print("Exercise 1 correct")

## Exercise 2 - a new tally and a run size

Still on the same `inp` object, add a neutron flux tally on the **outboard magnet**, cells
86 to 105:

1. tally number **400**, particle `n`, on those cells, with a **total** bin;
2. energy grid `[1e-6, 0.1, 1.0, 5.0, 14.0]`;
3. a description of your choice;
4. set the number of histories to **5e7**;
5. write the model to `outputs/iter_1d_variant.i`.

**Hints**
* `add_F_tally(id, particles, cells, add_total=..., energies=..., description=...)` -
  `particles` is a **list**, e.g. `["n"]`;
* `add_stopCard(5e7)`;
* `inp.write(OUT / "iter_1d_variant.i")`.

In [ ]:
# 1: the cells of the outboard magnet
magnet_cells = list(range(86, 106))

# 2: the new tally
energy_grid = [1e-6, 0.1, 1.0, 5.0, 14.0]
inp.add_F_tally(
    400,
    ["n"],
    magnet_cells,
    energies=energy_grid,
    description="Neutron flux in the outboard magnet [#/cm^2]",
    add_total=True,
)

# 3: how many histories we want
inp.add_stopCard(5e7)

# 4: write the modified model
inp.write(OUT / "iter_1d_variant.i")

print(inp.other_data["F400"])

In [ ]:
# --- self-check, do not modify ---
reloaded = Input.from_input(OUT / "iter_1d_variant.i")
keys = {k.upper(): str(v) for k, v in reloaded.other_data.items()}
f400 = "".join(v for k, v in keys.items() if k.startswith("F400"))

assert 400 in reloaded.tally_keys, f"tally 400 is missing, found {reloaded.tally_keys}"
assert "E400" in keys, "the energy grid card E400 is missing"
assert "1.4000e+01" in keys["E400"], "the energy grid does not reach 14 MeV"
assert "105" in f400, "cell 105 is not in the tally"
assert "50000000" in keys["NPS"], f"NPS is {keys['NPS']}"
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| what is in material X? | `mat_section["M7"].get_info()`, `.to_text()` |
| which libraries have this nuclide? | `LibManager.check4zaid` |
| what is the natural composition of Fe? | `LibManager.expand_element` |
| build a material from scratch | `Material.from_zaids` |
| smear two materials together | `mat_section.generate_material` |
| move the model to another library | `translate` |
| atom fraction <-> mass fraction | `switch_fraction` |
| mass density <-> atom density | `get_tad`, `get_density` |
| swap a material everywhere | `replace_material` |
| add / remove a tally | `add_F_tally`, `remove_tallies` |
| set the run size | `add_stopCard` |
| set up a void check | `prepare_void_check` |

The input is ready. In the next notebook the run has been done and we look at what came out of
it: **`03_parsing_outp_and_mctal.ipynb`**.